In [1]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & API AUTHENTICATION
# Run this cell first to install required libraries and authenticate.
# ==============================================================================
!pip install -q -U google-genai chromadb pypdf fpdf langchain-text-splitters numpy

import os
import getpass
import numpy as np
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from pypdf import PdfReader
from fpdf import FPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Official Google GenAI SDK
from google import genai
from google.genai import types

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion
# ------------------------------------------------------------------------------
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━

In [2]:
# ==============================================================================
# CELL 1: GENERATE SYNTHETIC KNOWLEDGE BASE (PDF)
# Run this cell to create a dummy HR Policy PDF for our live ingestion demo.
# ==============================================================================
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

document_text = """Acme Corp Global Employee Handbook (2026)

1. Remote Work Policy:
Employees may work remotely up to 3 days per week. Tuesdays and Thursdays are designated 'Anchor Days' where in-office presence is mandatory for collaborative meetings. Core working hours are 10:00 AM to 3:00 PM local time.

2. Annual Leave and PTO:
Full-time employees accrue 20 days of Paid Time Off (PTO) annually. Up to 5 unused PTO days can be rolled over to the next calendar year. Rollover days expire on March 31st.

3. Cloud Infrastructure Security:
All database access requires VPN connection and Multi-Factor Authentication (MFA). Database passwords must be rotated every 60 days. API keys must never be hardcoded into Git repositories.

4. Expense Reimbursements:
Business travel expenses must be submitted within 30 days of the trip via the Concur portal. Meal allowances are capped at $75 per diem. Alcohol is not a reimbursable expense.

5. Hardware Upgrades:
Engineers are eligible for a laptop refresh every 3 years. The standard issue is a 16-inch M-series MacBook Pro. Monitor allowance for home offices is $500.
"""

for line in document_text.split('\n'):
    pdf.cell(200, 10, txt=line, ln=True, align='L')

pdf_filename = "Acme_HR_Policy_2026.pdf"
pdf.output(pdf_filename)
print(f"✅ Generated synthetic PDF document: '{pdf_filename}'")

✅ Generated synthetic PDF document: 'Acme_HR_Policy_2026.pdf'


In [3]:
# ==============================================================================
# SECTION 1: EMBEDDINGS & SIMILARITY METRICS
# ==============================================================================
"""
1. EMBEDDINGS FOR RETRIEVAL:
   - An embedding converts raw unstructured text into a dense array of floating-point numbers (e.g., 768 dimensions for Gemini text-embedding-004).
   - In this high-dimensional space, semantically similar sentences are physically closer together.
   - Example: "The dog barked" and "A puppy yelped" have no overlapping keywords, but their embeddings will have high similarity.

2. SIMILARITY METRICS:
   - Cosine Similarity: Measures the angle between two vectors (1.0 = identical direction, 0.0 = orthogonal).
     Standard for text because it ignores vector magnitude (document length).
   - Dot Product: Computes angle AND magnitude. If vectors are normalized (L2=1), Dot Product == Cosine Similarity.
   - Euclidean Distance (L2): Measures straight-line distance. Sensitive to document length.
"""

def compute_cosine_similarity(vec_a, vec_b):
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b)

# Fetch embeddings for a query and two documents
resp = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["Refund my money", "I want my cash back", "The server is down"]
)
vectors = [emb.values for emb in resp.embeddings]

sim_1 = compute_cosine_similarity(vectors[0], vectors[1])
sim_2 = compute_cosine_similarity(vectors[0], vectors[2])

print("=== 🧮 COSINE SIMILARITY DEMONSTRATION ===")
print(f"Query: 'Refund my money'")
print(f"Similarity to 'I want my cash back': {sim_1:.4f} (High Semantic Affinity)")
print(f"Similarity to 'The server is down':  {sim_2:.4f} (Low Semantic Affinity)")

=== 🧮 COSINE SIMILARITY DEMONSTRATION ===
Query: 'Refund my money'
Similarity to 'I want my cash back': 0.7256 (High Semantic Affinity)
Similarity to 'The server is down':  0.5584 (Low Semantic Affinity)


In [4]:
# ==============================================================================
# SECTION 2: VECTOR DATABASES & CHUNKING STRATEGIES
# ==============================================================================
"""
VECTOR DATABASES (Why PostgreSQL isn't enough):
- Standard databases execute exact-match queries (WHERE text = 'x').
- Vector DBs (Chroma, FAISS, Pinecone) execute Approximate Nearest Neighbor (ANN) searches (e.g., HNSW indexing).
- They find the "closest" vectors in milliseconds across millions of documents.

CHUNKING STRATEGIES (The #1 reason RAG fails is bad chunking!):
1. Fixed-Size Chunking: Slice text every N tokens. (Flaw: Cuts sentences in half, destroying context).
2. Recursive Character Splitter: Tries to split on paragraphs (\n\n), then sentences (.), then words ( ), ensuring semantic blocks stay intact.
3. Overlap: Always overlap chunks by 10-20% to prevent losing context across boundaries.
"""

# Let's define our Custom Gemini Embedding Function for ChromaDB
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=input
        )
        # Return a list of float arrays
        return [emb.values for emb in response.embeddings]

gemini_ef = GeminiEmbeddingFunction()
print("✅ Custom Gemini Embedding Function for ChromaDB registered.")

✅ Custom Gemini Embedding Function for ChromaDB registered.


/tmp/ipykernel_706/2595921960.py:26: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


In [5]:
# ==============================================================================
# SECTION 3: LIVE DEMO — END-TO-END DOCUMENT INGESTION PIPELINE
# ==============================================================================
# Step 1: Extract Text from PDF
reader = PdfReader("Acme_HR_Policy_2026.pdf")
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"

print("1. PDF Loaded & Parsed.")

# Step 2: Recursive Character Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Max characters per chunk
    chunk_overlap=40,    # Context overlap
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_text(raw_text)
print(f"2. Document Split into {len(chunks)} contextual chunks.")

# Step 3: Initialize ChromaDB and Ingest Data
chroma_client = chromadb.Client()

# Create or reset the collection
collection_name = "acme_hr_policy"
try:
    chroma_client.delete_collection(collection_name)
except:
    pass
collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=gemini_ef
)

# Prepare lists for Chroma
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(chunks):
    documents.append(chunk)
    ids.append(f"chunk_{i}")
    metadatas.append({"source": "Acme_HR_Policy_2026.pdf", "chunk_index": i})

# Add to Vector Database (This automatically calls the Gemini Embedding API)
print("3. Embedding chunks and indexing in ChromaDB... (Please wait ~3 seconds)")
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)
print("✅ Ingestion Complete! Vector DB is ready for queries.")

1. PDF Loaded & Parsed.
2. Document Split into 9 contextual chunks.
3. Embedding chunks and indexing in ChromaDB... (Please wait ~3 seconds)
✅ Ingestion Complete! Vector DB is ready for queries.


In [9]:
# ==============================================================================
# SECTION 4: LIVE DEMO — RETRIEVAL & SEMANTIC SEARCH
# ==============================================================================
# Now we act as the user asking a question
user_query = "What happens to my unused vacation days at the end of the year?"

print(f"👤 USER QUERY: '{user_query}'\n")

# Query ChromaDB (Embeds the query, does cosine similarity search, returns Top K)
results = collection.query(
    query_texts=[user_query],
    n_results=2 # Top-k retrieval
)

print("=== 🔍 VECTOR DB RETRIEVAL RESULTS ===")
for i, (doc, meta, distance) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0])):
    # Note: Chroma uses L2 distance by default. Lower is better/closer.
    print(f"\n--- Rank {i+1} (Distance: {distance:.4f}) ---")
    print(f"Source: {meta['source']} (Chunk {meta['chunk_index']})")
    print(f"Text Content:\n{doc}")

"""
THE NEXT STEP (RAG GENERATION):
In a full RAG pipeline, we take these retrieved chunks and inject them into the LLM prompt:
"Answer the user query using ONLY the following context: <chunk_1> <chunk_2>. Query: What happens..."
(We will cover the generation half deeply tomorrow!)
"""

👤 USER QUERY: 'What happens to my unused vacation days at the end of the year?'

=== 🔍 VECTOR DB RETRIEVAL RESULTS ===

--- Rank 1 (Distance: 0.5856) ---
Source: Acme_HR_Policy_2026.pdf (Chunk 3)
Text Content:
2. Annual Leave and PTO:
Full-time employees accrue 20 days of Paid Time Off (PTO) annually. Up to 5 unused PTO days can be rolled over to the next calendar year. Rollover days expire on March 31st.

--- Rank 2 (Distance: 0.9046) ---
Source: Acme_HR_Policy_2026.pdf (Chunk 8)
Text Content:
5. Hardware Upgrades:
Engineers are eligible for a laptop refresh every 3 years. The standard issue is a 16-inch M-series MacBook Pro. Monitor allowance for home offices is $500.


'\nTHE NEXT STEP (RAG GENERATION):\nIn a full RAG pipeline, we take these retrieved chunks and inject them into the LLM prompt:\n"Answer the user query using ONLY the following context: <chunk_1> <chunk_2>. Query: What happens..."\n(We will cover the generation half deeply tomorrow!)\n'

In [7]:
# ==============================================================================
# SECTION 5: RAG VS. FINE-TUNING DECISION FRAMEWORK
# ==============================================================================
"""
THE MOST COMMON INDUSTRY CONFUSION: "Should I Fine-Tune or use RAG?"

┌───────────────────────┬──────────────────────────────────┬─────────────────────────────────┐
│ Feature               │ RAG (Retrieval-Augmented)        │ Fine-Tuning (SFT)               │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Primary Goal          │ Injecting specific, factual data │ Changing model tone, format, or │
│                       │ and reducing hallucinations.     │ teaching a new domain language. │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Data Freshness        │ Real-time. Update the Vector DB  │ Static. Model weights must be   │
│                       │ and the model knows it instantly.│ retrained to learn new facts.   │
├───────────────────────┼──────────────────────┼─────────────────────────────────────────────┤
│ Hallucination Control │ High. Grounded by strict prompt  │ Low/Medium. Prone to mixing up  │
│                       │ constraints and citations.       │ facts embedded in weights.      │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Cost & Speed to Build │ Fast & Cheap (Days/Weeks).       │ Slow & Expensive (Weeks/Months).│
└───────────────────────┴──────────────────────────────────┴─────────────────────────────────┘

RULE OF THUMB:
1. If the model needs to know NEW FACTS -> Use RAG.
2. If the model needs to learn a NEW SKILL or TONE -> Use Fine-Tuning.
3. If both -> Do Both (Fine-Tune for format, RAG for facts).
"""

'\nTHE MOST COMMON INDUSTRY CONFUSION: "Should I Fine-Tune or use RAG?"\n\n┌───────────────────────┬──────────────────────────────────┬─────────────────────────────────┐\n│ Feature               │ RAG (Retrieval-Augmented)        │ Fine-Tuning (SFT)               │\n├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤\n│ Primary Goal          │ Injecting specific, factual data │ Changing model tone, format, or │\n│                       │ and reducing hallucinations.     │ teaching a new domain language. │\n├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤\n│ Data Freshness        │ Real-time. Update the Vector DB  │ Static. Model weights must be   │\n│                       │ and the model knows it instantly.│ retrained to learn new facts.   │\n├───────────────────────┼──────────────────────┼─────────────────────────────────────────────┤\n│ Hallucination Control │ High. Grounded by strict prompt  │ 

In [13]:
# ==============================================================================
# SECTION 6: STUDENT GRADED LAB — SOLUTION
# ==============================================================================

# 1. Chunking
# ------------------------------------------------------------------------------
# Import the RecursiveCharacterTextSplitter from LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter with the required settings
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=25
)

# Split the IT knowledge base into smaller chunks
chunks = text_splitter.split_text(it_knowledge_base)

# Display the generated chunks
print("=== CHUNKS ===")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}:")
    print(chunk)


# 2. Database Creation & Ingestion
# ------------------------------------------------------------------------------
# Create a NEW ChromaDB collection called "it_support_kb"
it_collection = chroma_client.get_or_create_collection(
    name="it_support_kb"
)


# 3. Add to Collection
# ------------------------------------------------------------------------------
# Create unique IDs for every chunk
ids = [f"it_chunk_{i}" for i in range(len(chunks))]

# Create metadata for every chunk
metadatas = [
    {
        "source": "IT_Support_Troubleshooting_Guide",
        "chunk_index": i
    }
    for i in range(len(chunks))
]

# Add the chunks, IDs, and metadata to ChromaDB
it_collection.add(
    documents=chunks,
    ids=ids,
    metadatas=metadatas
)

print("\n✅ IT Support knowledge base successfully added to ChromaDB!")


# 4. Test Queries
# ------------------------------------------------------------------------------
# Define the 3 required test queries
test_queries = [
    "Why does my VPN connection keep timing out?",
    "How do I install the 3rd floor marketing printer?",
    "How long does an account lockout last?"
]

# Run each query and print the top 1 result
for query in test_queries:

    print(f"\n\n👤 QUERY: {query}")

    # Query ChromaDB and retrieve only the best matching chunk
    results = it_collection.query(
        query_texts=[query],
        n_results=1
    )

    # Extract the top result
    top_document = results["documents"][0][0]
    top_metadata = results["metadatas"][0][0]
    top_distance = results["distances"][0][0]

    # Print the result
    print(f"🔍 Distance: {top_distance:.4f}")
    print(
        f"📄 Source: {top_metadata['source']} "
        f"(Chunk {top_metadata['chunk_index']})"
    )
    print(f"📌 Top Result:\n{top_document}")

=== CHUNKS ===

Chunk 0:
IT Support Troubleshooting Guide:

Chunk 1:
1. VPN Connection Failures:

Chunk 2:
If Cisco AnyConnect fails with 'Connection Timeout', ensure the user is not on a public hotel Wi-Fi network that blocks UDP port 443. Instruct the

Chunk 3:
port 443. Instruct the user to switch to a mobile hotspot.

Chunk 4:
2. Printer Installation:

Chunk 5:
To map the 3rd floor marketing printer, users must connect to the corporate network, open File Explorer, type '\printserver01\MKTG_Color_Laser', and

Chunk 6:
and double-click to install drivers automatically.

Chunk 7:
3. Account Lockouts:

Chunk 8:
Active Directory accounts lock out after 5 invalid password attempts. The lockout duration is 30 minutes. Helpdesk agents can manually unlock

Chunk 9:
can manually unlock accounts via the Azure Admin Portal after verifying the employee's PIN.


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 39.5MiB/s]



✅ IT Support knowledge base successfully added to ChromaDB!


👤 QUERY: Why does my VPN connection keep timing out?
🔍 Distance: 0.5577
📄 Source: IT_Support_Troubleshooting_Guide (Chunk 1)
📌 Top Result:
1. VPN Connection Failures:


👤 QUERY: How do I install the 3rd floor marketing printer?
🔍 Distance: 0.5528
📄 Source: IT_Support_Troubleshooting_Guide (Chunk 5)
📌 Top Result:
To map the 3rd floor marketing printer, users must connect to the corporate network, open File Explorer, type '\printserver01\MKTG_Color_Laser', and


👤 QUERY: How long does an account lockout last?
🔍 Distance: 0.5430
📄 Source: IT_Support_Troubleshooting_Guide (Chunk 7)
📌 Top Result:
3. Account Lockouts:
